# **Modernized Sign Language Recognition Model Training**
This notebook replaces the deprecated `mediapipe-model-maker` workflow. It uses standard MediaPipe to extract hand landmarks from your image dataset and trains a custom Keras classifier, exporting it directly to TensorFlow.js format for your React app.

In [1]:
# Install modern, stable dependencies
!pip install --upgrade pip
!pip install mediapipe tensorflow tensorflowjs opencv-python matplotlib

# **Import Libraries**

In [2]:
import os
import cv2
import numpy as np
import mediapipe as mp
import tensorflow as tf
import matplotlib.pyplot as plt
from tensorflow.keras import layers, models
from sklearn.model_selection import train_test_split
from google.colab import files

# **Connect to Google Drive Dataset**

In [3]:
from google.colab import drive
drive.mount('/content/drive')

my_folder_path = '/content/drive/MyDrive/Dataset'

Mounted at /content/drive


In [4]:
# Get and verify sorted labels
labels = sorted([i for i in os.listdir(my_folder_path) if os.path.isdir(os.path.join(my_folder_path, i))])
print(f"Total classes: {len(labels)}")
print("Classes found:", labels)

Total classes: 52
Classes found: ['0', '1', '2', '3', '4', '5', '6', '7', '8', '9', 'A', 'B', 'Bye', 'C', 'D', 'Deaf', 'E', 'F', 'G', 'H', 'Hello', 'I', 'ILoveYou', 'J', 'K', 'L', 'Learn', 'M', 'Me', 'Meet', 'N', 'Name', 'No', 'NotOk', 'O', 'Ok', 'P', 'Pen', 'Please', 'Q', 'R', 'S', 'T', 'Tell', 'Thankyou', 'U', 'V', 'W', 'X', 'Y', 'Yes', 'none']


# **Extract Landmarks from Images (Data Preparation)**
We use MediaPipe's base Hand Landmarker framework to parse all images into coordinate matrices ($21 \text{ landmarks} \times 3 \text{ dimensions} = 63 \text{ features}$). This completely removes the image loading bugs present in the old library.

In [5]:
import mediapipe as mp
from mediapipe.tasks import python
from mediapipe.tasks.python import vision

# 1. Configure the Hand Landmarker Options
if not os.path.exists('hand_landmarker.task'):
    print("Downloading baseline hand landmarker model...")
    !wget -q https://storage.googleapis.com/mediapipe-models/hand_landmarker/hand_landmarker/float16/1/hand_landmarker.task

base_options = python.BaseOptions(model_asset_path='hand_landmarker.task')
options = vision.HandLandmarkerOptions(
    base_options=base_options,
    running_mode=vision.RunningMode.IMAGE,
    num_hands=1,
    min_hand_detection_confidence=0.5
)

def normalize_landmarks(hand_landmarks):
    """Make landmarks translation- and scale-invariant.
    Wrist (landmark 0) becomes the origin; scale is normalized by the
    wrist -> middle-finger-MCP (landmark 9) distance. This way the model
    learns hand SHAPE, not where the hand happened to be in the frame.
    """
    coords = np.array([[lm.x, lm.y, lm.z] for lm in hand_landmarks])
    wrist = coords[0].copy()
    coords -= wrist
    scale = np.linalg.norm(coords[9])
    if scale > 1e-6:
        coords /= scale
    return coords.flatten()

X = []
y = []

# Track detection success/failure per class so we can see whether any
# class is losing a large chunk of its (already small) sample count.
detection_log = {}

print("Starting landmark extraction using modern MediaPipe Tasks API...")

# 2. Open the landmarker instance
with vision.HandLandmarker.create_from_options(options) as landmarker:
    for label_idx, label in enumerate(labels):
        label_dir = os.path.join(my_folder_path, label)
        if not os.path.isdir(label_dir):
            continue

        image_files = os.listdir(label_dir)
        detected, missed = 0, 0

        for img_name in image_files:
            img_path = os.path.join(label_dir, img_name)

            try:
                mp_image = mp.Image.create_from_file(img_path)
            except Exception:
                missed += 1
                continue

            detection_result = landmarker.detect(mp_image)

            if detection_result.hand_landmarks:
                first_hand_landmarks = detection_result.hand_landmarks[0]
                X.append(normalize_landmarks(first_hand_landmarks))
                y.append(label_idx)
                detected += 1
            else:
                missed += 1

        detection_log[label] = (detected, missed)
        print(f"'{label}': {detected} detected / {missed} missed (of {len(image_files)} images)")

X = np.array(X)
y = np.array(y)
print(f"\nExtraction Complete! Formatted dataset shape: {X.shape}")

# Flag classes that ended up with very few usable samples -- these are
# the ones most likely to give unreliable / inconsistent predictions.
print("\n--- Classes with fewer than 30 detected samples (high risk) ---")
for label, (detected, missed) in sorted(detection_log.items(), key=lambda kv: kv[1][0]):
    if detected < 30:
        print(f"  {label}: only {detected} usable samples")


Starting landmark extraction using modern MediaPipe Tasks API...
'0': 150 detected / 55 missed (of 205 images)
'1': 158 detected / 48 missed (of 206 images)
'2': 132 detected / 74 missed (of 206 images)
'3': 198 detected / 8 missed (of 206 images)
'4': 181 detected / 26 missed (of 207 images)
'5': 207 detected / 0 missed (of 207 images)
'6': 161 detected / 46 missed (of 207 images)
'7': 161 detected / 45 missed (of 206 images)
'8': 174 detected / 34 missed (of 208 images)
'9': 203 detected / 1 missed (of 204 images)
'A': 60 detected / 0 missed (of 60 images)
'B': 47 detected / 1 missed (of 48 images)
'Bye': 86 detected / 0 missed (of 86 images)
'C': 49 detected / 0 missed (of 49 images)
'D': 40 detected / 7 missed (of 47 images)
'Deaf': 32 detected / 0 missed (of 32 images)
'E': 47 detected / 2 missed (of 49 images)
'F': 48 detected / 0 missed (of 48 images)
'G': 41 detected / 7 missed (of 48 images)
'H': 16 detected / 0 missed (of 16 images)
'Hello': 59 detected / 1 missed (of 60 imag

# **Train/Test Split & One-Hot Encoding**

In [6]:
# Categorical encoding
num_classes = len(labels)
y_encoded = tf.keras.utils.to_categorical(y, num_classes=num_classes)

# 80/10/10 Split matching original requirements
X_train, X_rest, y_train, y_rest = train_test_split(X, y_encoded, test_size=0.2, random_state=42)
X_val, X_test, y_val, y_test = train_test_split(X_rest, y_rest, test_size=0.5, random_state=42)

print(f"Train samples: {X_train.shape[0]}, Validation: {X_val.shape[0]}, Test: {X_test.shape[0]}")

Train samples: 2904, Validation: 363, Test: 364


# **Balance Training Data (fix class imbalance)**
Several classes (e.g. K, M, H, N, P, Q...) have far fewer images than others (digits, `none`). We oversample + jitter-augment the minority classes in *landmark space* so the model sees roughly equal examples per class. This is only applied to the training split -- validation/test stay untouched so the reported accuracy is honest.

In [7]:
from collections import Counter

def augment_landmarks(vec, n_variants=1, noise_std=0.02, rot_std_deg=8, scale_std=0.05):
    """Create slightly jittered copies of a normalized landmark vector:
    small random rotation (around z), scale jitter, and coordinate noise.
    Helps the small classes generalize instead of memorizing 12-13 samples."""
    pts = vec.reshape(-1, 3)
    variants = []
    for _ in range(n_variants):
        theta = np.radians(np.random.normal(0, rot_std_deg))
        rot = np.array([[np.cos(theta), -np.sin(theta), 0],
                         [np.sin(theta),  np.cos(theta), 0],
                         [0, 0, 1]])
        p = pts @ rot.T
        p = p * np.random.normal(1.0, scale_std)
        p = p + np.random.normal(0, noise_std, p.shape)
        variants.append(p.flatten())
    return variants

# Figure out target sample count per class = size of the largest class
train_labels = np.argmax(y_train, axis=1)
counts = Counter(train_labels)
target = max(counts.values())

X_bal, y_bal = [X_train[i] for i in range(len(X_train))], [y_train[i] for i in range(len(y_train))]

for cls, cnt in counts.items():
    if cnt >= target:
        continue
    cls_indices = np.where(train_labels == cls)[0]
    needed = target - cnt
    for k in range(needed):
        src_idx = cls_indices[k % len(cls_indices)]
        aug_vec = augment_landmarks(X_train[src_idx], n_variants=1)[0]
        X_bal.append(aug_vec)
        y_bal.append(y_train[src_idx])

X_train_bal = np.array(X_bal)
y_train_bal = np.array(y_bal)

print("Before balancing:", dict(Counter(train_labels)))
print(f"After balancing: {X_train_bal.shape[0]} training samples, "
      f"{target} per class (target)")


Before balancing: {np.int64(51): 139, np.int64(23): 38, np.int64(8): 130, np.int64(3): 157, np.int64(44): 77, np.int64(5): 168, np.int64(32): 64, np.int64(1): 113, np.int64(10): 50, np.int64(2): 101, np.int64(4): 145, np.int64(12): 71, np.int64(7): 134, np.int64(9): 168, np.int64(17): 36, np.int64(25): 47, np.int64(31): 41, np.int64(28): 44, np.int64(47): 45, np.int64(0): 117, np.int64(13): 40, np.int64(6): 123, np.int64(50): 65, np.int64(37): 36, np.int64(20): 47, np.int64(48): 13, np.int64(43): 49, np.int64(11): 35, np.int64(49): 38, np.int64(38): 39, np.int64(14): 32, np.int64(16): 42, np.int64(34): 36, np.int64(33): 41, np.int64(29): 43, np.int64(35): 46, np.int64(21): 40, np.int64(18): 33, np.int64(15): 24, np.int64(24): 14, np.int64(41): 15, np.int64(42): 13, np.int64(30): 9, np.int64(46): 13, np.int64(39): 13, np.int64(27): 13, np.int64(22): 29, np.int64(45): 15, np.int64(19): 13, np.int64(26): 25, np.int64(36): 12, np.int64(40): 13}
After balancing: 8736 training samples, 168 p

# **Build & Train the Classifier Model**

In [8]:
model = models.Sequential([
    layers.Dense(128, activation='relu', input_shape=(63,)),
    layers.BatchNormalization(),
    layers.Dropout(0.2),
    layers.Dense(64, activation='relu'),
    layers.Dropout(0.1),
    layers.Dense(num_classes, activation='softmax')
])

model.summary()

model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=0.001),
    loss='categorical_crossentropy',
    metrics=['categorical_accuracy']
)

# EarlyStopping so we don't overfit now that minority classes have
# repeated/augmented samples.
early_stop = tf.keras.callbacks.EarlyStopping(
    monitor='val_categorical_accuracy', patience=6, restore_best_weights=True
)

# Train on the BALANCED training set (X_train_bal/y_train_bal) instead
# of the raw imbalanced X_train/y_train.
history = model.fit(
    X_train_bal, y_train_bal,
    validation_data=(X_val, y_val),
    epochs=60,
    batch_size=32,
    callbacks=[early_stop]
)


/usr/local/lib/python3.12/dist-packages/keras/src/layers/core/dense.py:106: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ dense (Dense)                   │ (None, 128)            │         8,192 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization             │ (None, 128)            │           512 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout (Dropout)               │ (None, 128)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ (None, 64)             │         8,256 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_1 (Dropout)             │ (None, 64)             │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_2 (Dense)                 │ (None, 52)             │         3,380 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 20,340 (79.45 KB)

 Trainable params: 20,084 (78.45 KB)

 Non-trainable params: 256 (1.00 KB)

Epoch 1/60
273/273 ━━━━━━━━━━━━━━━━━━━━ 5s 7ms/step - categorical_accuracy: 0.5076 - loss: 2.0518 - val_categorical_accuracy: 0.8926 - val_loss: 1.2075
Epoch 2/60
273/273 ━━━━━━━━━━━━━━━━━━━━ 1s 4ms/step - categorical_accuracy: 0.8482 - loss: 0.6143 - val_categorical_accuracy: 0.9394 - val_loss: 0.3495
Epoch 3/60
273/273 ━━━━━━━━━━━━━━━━━━━━ 1s 4ms/step - categorical_accuracy: 0.8994 - loss: 0.3917 - val_categorical_accuracy: 0.9394 - val_loss: 0.2534
Epoch 4/60
273/273 ━━━━━━━━━━━━━━━━━━━━ 1s 4ms/step - categorical_accuracy: 0.9193 - loss: 0.2958 - val_categorical_accuracy: 0.9477 - val_loss: 0.2111
Epoch 5/60
273/273 ━━━━━━━━━━━━━━━━━━━━ 1s 5ms/step - categorical_accuracy: 0.9326 - loss: 0.2446 - val_categorical_accuracy: 0.9421 - val_loss: 0.2145
Epoch 6/60
273/273 ━━━━━━━━━━━━━━━━━━━━ 1s 3ms/step - categorical_accuracy: 0.9417 - loss: 0.2043 - val_categorical_accuracy: 0.9614 - val_loss: 0.1617
Epoch 7/60
273/273 ━━━━━━━━━━━━━━━━━━━━ 1s 3ms/step - categorical_accuracy: 0.9432 - los

# **Evaluate Model**

In [9]:
loss, acc = model.evaluate(X_test, y_test)
print(f"Test Loss: {loss:.4f}, Test Accuracy: {acc:.4f}")

12/12 ━━━━━━━━━━━━━━━━━━━━ 1s 46ms/step - categorical_accuracy: 0.9533 - loss: 0.1854
Test Loss: 0.1854, Test Accuracy: 0.9533


# **Export Directly to TensorFlow.js Format**
This compiles and prepares the asset files into a JSON structure compatible with your React frontend setup.

In [10]:
model.save('keras_gesture_model.h5')
!tensorflowjs_converter --input_format=keras keras_gesture_model.h5 ./tfjs_model
!zip -r gesture_classifier.zip ./tfjs_model
from google.colab import files
files.download('gesture_classifier.zip')

2026-07-14 08:06:55.074657: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1784016415.099625    9584 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1784016415.107366    9584 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1784016415.124872    9584 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1784016415.124916    9584 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1784016415.124921    9584 computation_placer.cc:177] computation placer alr

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>